In [4]:
import numpy as np
from PIL import Image

In [5]:
def clip_image(img_float):
    return np.clip(img_float, 0, 255).astype(np.uint8)

# Operações algebricas
    # Dissolve uniforme
def dissolve_uniforme(f, g, t): # (1 - t)*f + t*g
    f_d = f.astype(np.float64)
    g_d = g.astype(np.float64)
    out = (1.0 - t) * f_d + t * g_d
    return clip_image(out);

    # Dissolve nao uniforme
def dissolve_nao_uniforme(f, g, mapa_t): # mapa_t é matriz de pesos entre 0 e 1
    f_d = f.astype(np.float64)
    g_d = g.astype(np.float64)
    mapa_d = mapa_t.astype(np.float64)
    out = (1.0 - mapa_d) * f_d + mapa_d * g_d
    return clip_image(out);


### Transformações de intensidade

In [6]:
# Transformações de intensidade

def negativo(f): # inverte escala de cinza
    return 255 - f

# Estica a faixa [r_min, r_max] pra [s_min, s_max]
def alargamento_contraste(f, r_min, r_max, s_min = 0, s_max = 255):
    f_d = f.astype(np.float64)
    out = (f_d - r_min) * ((r_max - r_min) / (s_max - s_min)) + s_min
    return clip_image(out);

def limiarizacao(f, limiar): # Binariza a imagem, pixels com valor >= limiar recebem 255, caso contrario recebem 0
    out = np.where(f >= limiar, 255, 0)
    return out.astype(np.uint8)

def potencia_gamma(f, c, gamma): #Transformação de potência: s = c * r^gamma
    f_norm = f.astype(np.float64) / 255
    out = c * (f_norm ** gamma) * 255
    return out.astype(np.uint8)

def logaritmo(f, c = 1.0):
    f_norm = f.astype(np.float64) / 255
    out = c * np.log(1.0 + f_norm) * (255 / np.log(2.0))
    return clip_image(out)



### Histograma

In [8]:
# Histograma
def expansao(f):
    r_min = float(np.min(f))
    r_max = float(np.max(f))

    if(r_max == r_min):
        return f.copy()
    return alargamento_contraste(f, r_min, r_max, 0, 255);

def equalizacao(f):
    altura, largura = f.shape # Altura e Largura da imagem
    n_pixels = altura * largura

    hist = np.zeros(256, dtype= int)
    for val in f.flat:
        hist[val] += 1

    cdf = np.cumsum(hist)
    t_map = np.round(255 * cdf / n_pixels).astype(np.uint8)
    
    out = t_map[f]

    return out


### Contraste adaptativo

In [ ]:
# Contraste adaptativo
def contraste_adaptivo(f, n, c):
    pad = n // 2
    f_d = f.astype(np.float64)
    f_padded = np.pad(f_d, pad, mode = 'edge')
    alt, larg = f.shape

    out = np.zeros_like(f_d)

    for i in range(alt):
        for j in range(larg):
            regiao = f_padded[i : i + n, j + n]
            m_local = np.mean(regiao)

            out[i, j] = c * (f_d[i, j] - m_local) + m_local

    return clip_image(out)

### Filtragem espacial